In [11]:
from SCSCtrl import TTLServo
import time

print("Servo controller loaded.")

Servo controller loaded.


In [12]:
# Claw servo ID
CLAW_SERVO_ID = 4

# Claw angles
CLAW_OPEN = -10
CLAW_CLOSED = -75

# Arm positions
ARM_HOME_X = 130
ARM_HOME_Y = 20

ARM_DOWN_X = 150
ARM_DOWN_Y = -138

ARM_UP_X = 120
ARM_UP_Y = 45

In [14]:
TTLServo.servoAngleCtrl(
    CLAW_SERVO_ID,
    CLAW_OPEN,
    1,
    150
)

print("Claw Open")

Claw Open


In [4]:
TTLServo.servoAngleCtrl(
    CLAW_SERVO_ID,
    CLAW_CLOSED,
    1,
    150
)

print("Claw Closed")

Claw Closed


In [5]:
TTLServo.xyInputSmooth(
    ARM_HOME_X,
    ARM_HOME_Y,
    0.8
)

print("Arm Home")

Arm Home


In [6]:
TTLServo.xyInputSmooth(
    ARM_DOWN_X,
    ARM_DOWN_Y,
    0.8
)

print("Arm Down")

Arm Down


In [7]:
TTLServo.xyInputSmooth(
    ARM_UP_X,
    ARM_UP_Y,
    0.8
)

print("Arm Up")

Arm Up


### Full pick-up Sequence

In [8]:
print("Starting pickup test")

# Step 1 - go home first
TTLServo.xyInputSmooth(ARM_HOME_X, ARM_HOME_Y, 0.5)
time.sleep(2)

# Step 2 - open claw
TTLServo.servoAngleCtrl(4, CLAW_OPEN, 1, 150)
time.sleep(1)

# Step 3 - lower to cap
TTLServo.xyInputSmooth(ARM_DOWN_X, ARM_DOWN_Y, 0.5)
time.sleep(2)

# Step 4 - close claw
TTLServo.servoAngleCtrl(4, CLAW_CLOSED, 1, 150)
time.sleep(2)

# Step 5 - return home with cap secured
TTLServo.xyInputSmooth(ARM_HOME_X, ARM_HOME_Y, 0.5)
time.sleep(2)

print("Pickup sequence complete")

Starting pickup test
Pickup sequence complete


### Drop Sequence

In [9]:
# New variable to add at the top with the others
DROP_X = 130  # tune physically
DROP_Y = 20   # tune physically

print("Starting drop test")

# Step 1 - ensure arm is at home before moving to container
TTLServo.xyInputSmooth(ARM_HOME_X, ARM_HOME_Y, 0.8)
time.sleep(2)

# Step 2 - move above container (NOT ARM_DOWN which is floor level)
TTLServo.xyInputSmooth(DROP_X, DROP_Y, 0.8)
time.sleep(2)

# Step 3 - release
TTLServo.servoAngleCtrl(CLAW_SERVO_ID, CLAW_OPEN, 1, 150)
time.sleep(1)

# Step 4 - return home
TTLServo.xyInputSmooth(ARM_HOME_X, ARM_HOME_Y, 0.8)
time.sleep(2)  # this was missing

print("Drop complete")

Starting drop test
Drop complete


### Automatic Repeating
Calibration

In [10]:
for _ in range(5):  # patched infinite loop

    print("OPEN")
    TTLServo.servoAngleCtrl(4, CLAW_OPEN, 1, 150)
    time.sleep(2)

    print("CLOSE")
    TTLServo.servoAngleCtrl(4, CLAW_CLOSED, 1, 150)
    time.sleep(2)

OPEN
CLOSE
OPEN
CLOSE
OPEN
CLOSE
OPEN
CLOSE
OPEN
CLOSE


### Combined camera detection + claw delivery sequence

Run this cell on the JETANK after confirming the claw-only test cells work.


In [ ]:
"""
Live Roboflow bottle-cap detection + JETANK claw pick/drop sequence.

Flow:
1. Camera watches for bottle cap.
2. When a bottle cap is detected above CONFIDENCE_THRESHOLD:
   - arm goes home
   - claw opens
   - arm lowers to pickup position
   - claw closes to grab
   - arm lifts / returns home
   - waits to simulate driving to the grey box
   - arm moves above grey container
   - claw opens to release
   - arm returns home
3. Detection pauses while the claw sequence is running, so it will not trigger twice.

Controls:
  Q / ESC  stop early
  C        calibrate yellow boundary threshold to current measured distance
"""

import base64
import json
import sys
import time
from urllib import request, error

import cv2
import numpy as np

from SCSCtrl import TTLServo


# =========================
# Roboflow model settings
# =========================
API_KEY = "Ub1KVwtGHHdLLKRzoxdG"
API_URL = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"

CONFIDENCE_THRESHOLD = 0.8
INTERVAL = 3.0
TARGET_CLASS = "bottle cap"
CAMERA_INDEX = 0


# =========================
# Yellow boundary detection
# =========================
YELLOW_LOWER = (18, 80, 80)
YELLOW_UPPER = (38, 255, 255)
MIN_YELLOW_AREA = 150
BOUNDARY_THRESHOLD_PX = 224
REF_MODE = "bottom-center"


# =========================
# Claw / arm settings
# Values copied from Joaquin-Test2.ipynb
# Tune these physically on the real JETANK before autonomous runs.
# =========================
CLAW_SERVO_ID = 4

CLAW_OPEN = -10
CLAW_CLOSED = -75

ARM_HOME_X = 130
ARM_HOME_Y = 20

ARM_DOWN_X = 150
ARM_DOWN_Y = -138

ARM_UP_X = 120
ARM_UP_Y = 45

# Grey box / drop position.
# IMPORTANT: tune these to the real position above your grey container.
DROP_X = 130
DROP_Y = 20

# Movement timing
SERVO_SPEED = 0.8
CLAW_MOVE_TIME = 1.0
ARM_MOVE_TIME = 2.0

# This delay simulates the robot travelling to the grey box.
# Replace this later with real navigation code.
SIMULATED_TRAVEL_TO_BOX_SECONDS = 5.0


def move_arm(x, y, speed=SERVO_SPEED, wait=ARM_MOVE_TIME, label=None):
    """Move arm smoothly and wait for motion to finish."""
    if label:
        print(label)
    TTLServo.xyInputSmooth(x, y, speed)
    time.sleep(wait)


def set_claw(angle, wait=CLAW_MOVE_TIME, label=None):
    """Set claw servo angle and wait for motion to finish."""
    if label:
        print(label)
    TTLServo.servoAngleCtrl(CLAW_SERVO_ID, angle, 1, 150)
    time.sleep(wait)


def cap_pick_drop_sequence():
    """
    Complete claw action:
    pick cap -> hold/lift -> simulated travel -> drop in grey box -> home.
    """
    print("\n=== CAP DELIVERY SEQUENCE START ===")

    # Start safe
    move_arm(ARM_HOME_X, ARM_HOME_Y, 0.5, 2.0, "1) Arm to home")
    set_claw(CLAW_OPEN, 1.0, "2) Open claw")

    # Pick cap
    move_arm(ARM_DOWN_X, ARM_DOWN_Y, 0.5, 2.0, "3) Lower arm to bottle cap")
    set_claw(CLAW_CLOSED, 2.0, "4) Close claw / grab cap")

    # Lift and hold
    move_arm(ARM_UP_X, ARM_UP_Y, 0.8, 2.0, "5) Lift cap")
    move_arm(ARM_HOME_X, ARM_HOME_Y, 0.8, 2.0, "6) Return arm home while holding cap")

    # Simulated navigation to grey box
    print(f"7) Simulating travel to grey box for {SIMULATED_TRAVEL_TO_BOX_SECONDS:.1f}s")
    time.sleep(SIMULATED_TRAVEL_TO_BOX_SECONDS)

    # Drop into grey box
    move_arm(DROP_X, DROP_Y, 0.8, 2.0, "8) Move arm above grey container")
    set_claw(CLAW_OPEN, 1.0, "9) Release cap")

    # Return safe
    move_arm(ARM_HOME_X, ARM_HOME_Y, 0.8, 2.0, "10) Return arm home")

    print("=== CAP DELIVERY SEQUENCE COMPLETE ===\n")


def infer_frame(frame):
    ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError("Failed to JPEG-encode frame")

    image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")

    payload = {
        "api_key": API_KEY,
        "inputs": {
            "image": {"type": "base64", "value": image_b64},
            "confidence": CONFIDENCE_THRESHOLD,
        },
    }

    req = request.Request(
        url=API_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "python-urllib/3",
        },
        method="POST",
    )

    with request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))


def extract_predictions(result):
    outputs = result.get("outputs", [])
    if not outputs:
        return []
    preds = outputs[0].get("predictions", {})

    # Roboflow workflow nests predictions one level deep
    if isinstance(preds, dict):
        preds = preds.get("predictions", [])

    return preds or []


def draw_predictions(frame, predictions):
    for pred in predictions:
        x, y = pred["x"], pred["y"]
        w, h = pred["width"], pred["height"]
        x1, y1 = int(x - w / 2), int(y - h / 2)
        x2, y2 = int(x + w / 2), int(y + h / 2)
        label = f'{pred["class"]} {pred["confidence"]:.2f}'

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            label,
            (x1, max(y1 - 6, 12)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )
    return frame


def ref_point(frame):
    h, w = frame.shape[:2]
    if REF_MODE == "center":
        return w // 2, h // 2
    return w // 2, h - 1


def annotate_boundary(frame, threshold):
    """
    Detect yellow boundary, draw it, and return:
      (breach, distance_px)

    threshold is calibrated as the pixel distance equal to 2 cm.
    """
    rx, ry = ref_point(frame)
    cv2.drawMarker(frame, (rx, ry), (255, 255, 255), cv2.MARKER_CROSS, 16, 2)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(YELLOW_LOWER), np.array(YELLOW_UPPER))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    if cv2.countNonZero(mask) < MIN_YELLOW_AREA:
        cv2.putText(
            frame,
            "no boundary",
            (8, 44),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 200, 255),
            2,
            cv2.LINE_AA,
        )
        return False, None

    ys, xs = np.where(mask > 0)
    d2 = (xs - rx) ** 2 + (ys - ry) ** 2
    i = int(np.argmin(d2))

    nx, ny = int(xs[i]), int(ys[i])
    dist = float(np.sqrt(d2[i]))
    breach = dist < threshold
    color = (0, 0, 255) if breach else (0, 255, 0)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(frame, contours, -1, (0, 255, 255), 2)
    cv2.line(frame, (rx, ry), (nx, ny), color, 2)
    cv2.circle(frame, (nx, ny), 5, color, -1)

    label = f"{dist:.0f}px {'BREACH' if breach else 'ok'} (thr {threshold:.0f}=2cm)"
    cv2.putText(
        frame,
        label,
        (8, 44),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        color,
        2,
        cv2.LINE_AA,
    )

    return breach, dist


def run_inference(frame):
    """Returns (annotated_frame, predictions)."""
    try:
        result = infer_frame(frame)
    except error.HTTPError as e:
        print(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:300]}")
        return frame, []
    except Exception as e:
        print(f"Inference failed: {e}")
        return frame, []

    predictions = extract_predictions(result)

    if not predictions:
        print("No predictions.")
        return frame, []

    for i, p in enumerate(predictions):
        print(
            f"[{i}] {p['class']:14s} conf={p['confidence']:.3f} "
            f"x={p['x']:.0f} y={p['y']:.0f} "
            f"w={p['width']:.0f} h={p['height']:.0f}"
        )
    print()

    return draw_predictions(frame, predictions), predictions


def is_target_detected(predictions):
    """True when target class is detected with enough confidence."""
    for p in predictions:
        pred_class = str(p.get("class", "")).strip().lower()
        pred_conf = float(p.get("confidence", 0.0))

        if pred_class == TARGET_CLASS.lower() and pred_conf >= CONFIDENCE_THRESHOLD:
            return True

    return False


def probe_cameras(max_index=6):
    print("Probing cameras...")
    for idx in range(max_index):
        cap = cv2.VideoCapture(idx, cv2.CAP_AVFOUNDATION)
        ok = False

        if cap.isOpened():
            for _ in range(10):
                grabbed, frame = cap.read()
                if grabbed and frame is not None:
                    ok = True
                    break
                time.sleep(0.1)

        cap.release()
        print(f"  index {idx}: {'WORKS' if ok else '-'}")

    print("Run with an index, e.g.: python3 use_model_mac_claw.py 1")



def get_camera_index_from_argv(default_index=CAMERA_INDEX):
    """Return a camera index safely in both Jupyter and normal Python."""
    args = sys.argv[1:]

    if "--list" in args or "-l" in args:
        return "LIST"

    for arg in args:
        try:
            return int(arg)
        except ValueError:
            continue

    return default_index


def main():
    # Safe in both:
    #   normal script: python use_model_camera_claw.py 0
    #   Jupyter: ignores injected args like '-f kernel.json'
    cam_index = get_camera_index_from_argv(CAMERA_INDEX)

    if cam_index == "LIST":
        probe_cameras()
        return

    cap = cv2.VideoCapture(cam_index, cv2.CAP_AVFOUNDATION)

    if not cap.isOpened():
        raise SystemExit(
            f"Cannot open camera index {cam_index}. Run "
            "`python3 use_model_mac_claw.py --list` to see available cameras, or grant "
            "camera permission in System Settings > Privacy & Security > Camera."
        )

    # Warm up camera
    warm_ok = False
    for _ in range(30):
        ok, _frame = cap.read()
        if ok and _frame is not None:
            warm_ok = True
            break
        time.sleep(0.1)

    if not warm_ok:
        cap.release()
        raise SystemExit(
            "Camera opened but produced no frames. Close other apps using the "
            "camera and retry."
        )

    print(__doc__)
    print(f"Capturing every {INTERVAL:.0f}s, flagging '{TARGET_CLASS}'. Press Q to stop.\n")
    print("Calibration: place yellow line exactly 2cm from the white cross, then press C.\n")

    overlay = None
    last_call = 0.0
    breach_prev = False
    threshold = float(BOUNDARY_THRESHOLD_PX)
    last_dist = None
    miss = 0

    # Prevent repeated grabbing of the same cap.
    # Set to False after the sequence if you want continuous multi-cap pickup.
    cap_already_handled = False

    while True:
        ok, frame = cap.read()

        if not ok or frame is None:
            miss += 1
            if miss > 30:
                print("Camera stopped delivering frames.")
                break
            time.sleep(0.03)
            continue

        miss = 0
        now = time.time()

        if now - last_call >= INTERVAL and not cap_already_handled:
            last_call = now
            print(f"--- capture @ {time.strftime('%H:%M:%S')} ---")

            overlay, preds = run_inference(frame.copy())

            if is_target_detected(preds):
                print(f">>> '{TARGET_CLASS}' DETECTED")
                print("BOTTLE CAP DETECTED -> STARTING PICKUP")
                cap_already_handled = True

                # Stop camera display before moving the claw/arm.
                # This avoids repeated triggers while the arm is moving.
                cv2.imshow("Roboflow model - Mac camera", overlay)
                cv2.waitKey(1)

                cap_pick_drop_sequence()

                # Optional: after one cap, stop the script.
                # Comment these 2 lines if you want it to keep watching after dropping.
                print("One bottle cap handled. Press Q/ESC or close the window to stop.")
                # break

        display = (overlay if overlay is not None else frame).copy()

        breach, dist = annotate_boundary(display, threshold)
        last_dist = dist

        if breach and not breach_prev:
            print(
                f">>> BOUNDARY BREACH: {dist:.0f}px from yellow line "
                f"(threshold {threshold:.0f}px)"
            )

        breach_prev = breach

        cv2.imshow("Roboflow model - Mac camera", display)

        key = cv2.waitKey(30) & 0xFF

        if key in (ord("q"), 27):
            print("Stopped by user.")
            break

        elif key == ord("c"):
            if last_dist is None:
                print("Calibrate failed: no yellow line in view.")
            else:
                threshold = last_dist
                print(
                    f">>> CALIBRATED: 2cm = {threshold:.0f}px. "
                    f"To make it permanent, set BOUNDARY_THRESHOLD_PX = {threshold:.0f}"
                )

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
